# Pretrain a small language model with TrainLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Dhiraj309/TrainLM/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb)

Open a copy in Colab or import it into Kaggle from the button above. Choose a runtime with the hardware you need; the link does not provision a TPU.

The setup cell does not install or replace PyTorch/TorchTPU. Before a native TPU run, it reports installed `torch-tpu` and `libtpu` versions. TrainLM checks package compatibility before loading data.

Open this notebook on a **single-device native TorchTPU runtime**. The fresh-run cell builds a randomly initialized Hugging Face GPT-2, learns a tokenizer from training text, trains, evaluates, and exports the model. The default 10,000-token run is only a workflow smoke test, not a useful model or performance claim.

**Model architecture:** `model_type="gpt2"` selects Hugging Face GPT-2. TrainLM creates `GPT2LMHeadModel` with random weights; it does not load a pretrained GPT-2 checkpoint.

Run setup, then the fresh-run cell. To continue afterward, change `RUN_RESUME = False` to `True` in the separate resume cell and run **only that cell**. It validates the saved report and resumes from its checkpoint with a 12,000-token budget. Keep the same runtime, package versions, and full run directory; do not rerun setup or fresh training. After resume, rerun the artifact cell to bundle the updated report and export.

For a new independent run, use an unused `RUN_DIR`. Advanced settings and checkpoint recovery: [pretraining guide](../README.md#start-here-pretrain-a-language-model-from-scratch).


In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source],
    check=True,
)
requirements = [
    "huggingface-hub>=1.5,<2",
    "PyYAML>=6,<7",
    "transformers>=5.15,<6",
    "datasets>=3,<5",
    "tokenizers>=0.21,<1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")
from importlib.metadata import PackageNotFoundError, version

for package_name in ("torch-tpu", "libtpu"):
    try:
        print(f"{package_name}=={version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}=MISSING; native TPU needs a runtime that provides TorchTPU")


In [ ]:
from trainlm import TrainLMPretrainingConfig, run_pretraining
RUN_DIR = "trainlm-pretraining-fresh"
FRESH_MAX_TOKENS = 10_000  # Fresh run: max_tokens=10_000.
config = TrainLMPretrainingConfig(
    output_dir=RUN_DIR,
    max_tokens=FRESH_MAX_TOKENS,
    sequence_length=128,
    model_type="gpt2",  # Randomly initialized Hugging Face GPT-2.
    model_embedding_size=256,
    model_layers=4,
    model_attention_heads=4,
    accelerator="native_tpu",
)
result = run_pretraining(config, show_progress=True)
print(result.summary())
print(f"checkpoint: {result.checkpoint_path}")
print(f"report: {result.report_path}")
print(f"model: {result.export_path}")


In [ ]:
from dataclasses import replace
from trainlm import restore_pretraining_report, run_pretraining

RUN_RESUME = False  # Change to True after the fresh run, then run this cell only.
RESUME_MAX_TOKENS = 12_000
if RUN_RESUME:
    saved = restore_pretraining_report(f"{RUN_DIR}/report.json")
    resume_config = replace(saved.config, max_tokens=RESUME_MAX_TOKENS)
    resumed = run_pretraining(
        resume_config,
        resume_from_checkpoint=saved.checkpoint_path,
        extend_training_budget=True,
        show_progress=True,
        report_path=f"{RUN_DIR}/resume-report.json",
        export_path=f"{RUN_DIR}/resumed-export",
    )
    print(resumed.summary())
    print(f"resumed_from: {saved.checkpoint_path}")
    print(f"report: {resumed.report_path}")
    print(f"model: {resumed.export_path}")
else:
    print("Fresh run is the default. Set RUN_RESUME = True to continue.")


In [ ]:
from trainlm import bundle_pretraining_artifacts

if RUN_RESUME:
    bundle = bundle_pretraining_artifacts(resumed.report_path)
else:
    bundle = bundle_pretraining_artifacts(result.report_path)
try:
    from google.colab import files
except ImportError:
    print(f"Artifact bundle: {bundle}")
else:
    files.download(str(bundle))
